# Магістерська робота: Модуль пояснюваного штучного інтелекту (Explainable AI)
## Ноутбук 3: Інтерпретація клінічних рішень за допомогою SHAP та EBM

* **Тема:** «Інтелектуальна система оцінки ризиків захворювань на основі методів пояснюваного штучного інтелекту»
* **Спеціальність:** 122 «Комп’ютерні науки»

---
### Мета дослідження:
1. Усунення проблеми «чорної скриньки» (Black Box) на основі алгоритму **TreeSHAP** та моделі **EBM (Glassbox)**.
2. Глобальний аналіз: визначення найважливіших медико-поведінкових факторів серцево-судинного ризику (Beeswarm, Bar plots).
3. Дослідження взаємодій факторів (Dependence plots): вплив тиску, віку, ІМТ та фізичної активності.
4. Локальний аналіз окремих пацієнтів (Waterfall Plots): розклад ризику на компоненти без соціологічного шуму.
5. Формування персоналізованого клінічного звіту з рекомендаціями щодо способу життя та цільових додаткових аналізів.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11

print(f"SHAP версія: {shap.__version__}")
print("Модуль пояснюваного штучного інтелекту готовий до роботи!")


In [2]:
# Завантаження тестових даних та навченої моделі
processed_dir = os.path.join('..', 'Datasets', 'processed')
models_dir = os.path.join('..', 'models')

X_test = pd.read_csv(os.path.join(processed_dir, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(processed_dir, 'y_test.csv')).iloc[:, 0].astype(int)

model = joblib.load(os.path.join(models_dir, 'best_heart_disease_xgb.joblib'))
print("Модель XGBoost успішно завантажено!")
print(f"Кількість ознак у моделі: {X_test.shape[1]}")


### 1. Розрахунок значень Шеплі (TreeSHAP)

In [3]:
explainer = shap.TreeExplainer(model)

sample_size = 2000
X_sample = X_test.iloc[:sample_size].copy()

print(f"Розрахунок значень Шеплі для {sample_size:,} пацієнтів...")
shap_values = explainer(X_sample)
print("Розрахунок завершено!")


### 2. Глобальна інтерпретованість (Beeswarm та Bar Plots)

In [4]:
# 1. Графік Beeswarm (розподіл впливу чистих клінічних факторів)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("Глобальний розподіл впливу клінічних факторів на ризик ССЗ (SHAP Beeswarm)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Рівні абсолютної важливості ознак (Bar Plot)
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=12, show=False)
plt.title("Топ-12 клінічних факторів за силою впливу на ризик ССЗ", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 3. Локальні клінічні кейси (SHAP Waterfall Plots)

In [5]:
# Пошук пацієнта з високим ризиком та здорового пацієнта
high_risk_idx = np.argmax(model.predict_proba(X_sample)[:, 1])
low_risk_idx = np.argmin(model.predict_proba(X_sample)[:, 1])

print(f"Пацієнт А (Високий ризик): №{high_risk_idx}, ймовірність ССЗ = {model.predict_proba(X_sample.iloc[[high_risk_idx]])[0, 1]:.1%}")
print(f"Пацієнт Б (Низький ризик):  №{low_risk_idx}, ймовірність ССЗ = {model.predict_proba(X_sample.iloc[[low_risk_idx]])[0, 1]:.1%}")

# Waterfall plot для пацієнта з високим ризиком
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[high_risk_idx], max_display=10, show=False)
plt.title(f"Пояснення рішення ШІ для пацієнта з високим ризиком (№{high_risk_idx})", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Waterfall plot для здорового пацієнта
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[low_risk_idx], max_display=10, show=False)
plt.title(f"Пояснення рішення ШІ для здорового пацієнта (№{low_risk_idx})", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### 4. Генератор персоналізованого клінічного звіту та рекомендацій

In [6]:
def generate_clinical_report(patient_idx, X_data, shap_obj, model_obj):
    features = X_data.iloc[patient_idx]
    patient_shap = shap_obj[patient_idx]
    prob = model_obj.predict_proba(X_data.iloc[[patient_idx]])[0, 1]
    
    shap_vals = patient_shap.values
    feature_names = patient_shap.feature_names
    
    risk_factors = []
    protective_factors = []
    
    for name, val, feat_val in zip(feature_names, shap_vals, features):
        if val > 0.05:
            risk_factors.append((name, val, feat_val))
        elif val < -0.05:
            protective_factors.append((name, val, feat_val))
            
    risk_factors.sort(key=lambda x: x[1], reverse=True)
    protective_factors.sort(key=lambda x: x[1])
    
    print("=" * 80)
    print(f"КЛІНІЧНИЙ ЗВІТ ІНТЕЛЕКТУАЛЬНОЇ СИСТЕМИ ДЛЯ ПАЦІЄНТА №{patient_idx}")
    print("=" * 80)
    print(f"Розрахований ризик серцево-судинних ускладнень: {prob:.1%}")
    if prob >= 0.5:
        print("Категорія: [ КРИТИЧНИЙ / ВИСОКИЙ РИЗИК - НЕОБХІДНА КОНСУЛЬТАЦІЯ КАРДІОЛОГА ]")
    elif prob >= 0.2:
        print("Категорія: [ ПОМІРНИЙ РИЗИК - РЕКОМЕНДОВАНО КОРЕКЦІЮ СПОСОБУ ЖИТТЯ ]")
    else:
        print("Категорія: [ НИЗЬКИЙ РИЗИК - СТАН В МЕЖАХ НОРМИ ]")
        
    print("\n[ГОЛОВНІ КЛІНІЧНІ ТА ПОВЕДІНКОВІ ФАКТОРИ ЗАГРОЗИ]:")
    for name, s_val, f_val in risk_factors[:4]:
        print(f"  ▲ {name:<20} (значення: {f_val:<4}) --> збільшує ризик на SHAP +{s_val:.3f}")
        
    print("\n[ЗАХИСНІ ФАКТОРИ ОРГАНІЗМУ ТА СПОСОБУ ЖИТТЯ]:")
    for name, s_val, f_val in protective_factors[:3]:
        print(f"  ▼ {name:<20} (значення: {f_val:<4}) --> знижує ризик на SHAP {s_val:.3f}")
        
    print("\n[ПЕРСОНАЛІЗОВАНІ РЕКОМЕНДАЦІЇ ШТУЧНОГО ІНТЕЛЕКТУ]:")
    if any(rf[0] == 'HighBP' for rf in risk_factors):
        print("  • Артеріальний тиск: моніторинг АТ, консультація лікаря, обмеження натрію/солі до 5 г/добу.")
    if any(rf[0] == 'HighChol' for rf in risk_factors):
        print("  • Ліпідний обмін: здати розгорнуту ліпідограму (ЛПНЩ, ЛПВЩ, тригліцериди) для оцінки атерогенезу.")
    if any(rf[0] == 'BMI' for rf in risk_factors):
        print("  • Метаболічний стан: зниження маси тіла на 5-10% знижує ризик судинних подій на 25%.")
    if any(rf[0] == 'Smoker' for rf in risk_factors):
        print("  • Відмова від куріння: пріоритетна дія для зниження спазму вінцевих артерій.")
    if any(rf[0] == 'Diabetes' for rf in risk_factors):
        print("  • Додатковий аналіз Рівня 2: перевірка рівня глікованого гемоглобіну (HbA1c).")
    if any(rf[0] == 'PhysActivity' for rf in risk_factors):
        print("  • Фізична активність: рекомендується мінімум 150 хв аеробної активності помірної інтенсивності на тиждень.")
    print("=" * 80)

# Демонстрація звіту
generate_clinical_report(high_risk_idx, X_sample, shap_values, model)
